# 🔍 Notebook 02 — Eksploracja Danych
## Exploratory Data Analysis of Landmark Sequences

This notebook analyses the extracted landmark data before model training:
- Class distribution (normal vs. anomaly)
- Feature distributions and correlations
- Temporal patterns distinguishing falls from normal movement
- Visibility quality assessment

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
PALETTE = {'normal': '#4CAF50', 'anomaly': '#F44336'}
print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Option A: Load from CSV files
landmark_dir = Path('../data/landmarks')

if landmark_dir.exists() and any(landmark_dir.rglob('*.csv')):
    from src.data.loader import LandmarkLoader
    loader = LandmarkLoader()
    df = loader.load_directory(str(landmark_dir))
    print(f'Loaded {len(df):,} rows from CSV files.')
else:
    # Option B: Generate synthetic data
    print('No CSV data found — using synthetic dataset.')
    from src.data.generator import SyntheticGenerator
    gen = SyntheticGenerator(seed=42)
    df = gen.generate(n_normal=400, n_anomaly=200, seq_duration_s=3.0)
    print(f'Generated {len(df):,} synthetic landmark observations.')

print(f'\nColumns: {list(df.columns)}')
df.describe()

## 2. Class Distribution

In [ ]:
# Frame-level label distribution
frame_labels = df.groupby(['sequence_id', 'frame', 'label']).size().reset_index(name='n')
label_counts = frame_labels.groupby('label').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0d0d1a')

# Pie chart
axes[0].pie(
    label_counts.values,
    labels=['Normal (0)', 'Anomaly (1)'],
    colors=[PALETTE['normal'], PALETTE['anomaly']],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'color': 'white'},
    wedgeprops={'edgecolor': '#333', 'linewidth': 1.5},
)
axes[0].set_title('Class Distribution (Frames)', color='white')
axes[0].set_facecolor('#0d0d1a')

# By motion type
if 'motion_type' in df.columns:
    mt = df.groupby(['motion_type', 'label'])['frame'].nunique().reset_index(name='frames')
    colors_map = {0: PALETTE['normal'], 1: PALETTE['anomaly']}
    for lbl in [0, 1]:
        subset = mt[mt['label'] == lbl]
        axes[1].bar(
            subset['motion_type'],
            subset['frames'],
            color=colors_map[lbl],
            alpha=0.85,
            label='Normal' if lbl == 0 else 'Anomaly',
        )
    axes[1].set_title('Frames per Motion Type', color='white')
    axes[1].set_xlabel('Motion Type', color='#aaa')
    axes[1].set_ylabel('Frame Count', color='#aaa')
    axes[1].tick_params(colors='#aaa', axis='x', rotation=30)
    axes[1].legend()

for ax in axes:
    ax.set_facecolor('#1a1a2e')

plt.tight_layout()
Path('../reports').mkdir(exist_ok=True)
plt.savefig('../reports/class_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

## 3. Feature Engineering Preview

In [ ]:
from src.data.features import FeatureExtractor

fe = FeatureExtractor()
X, y = fe.frame_features(df, label_col='label')

print(f'Feature matrix: {X.shape}')
print(f'Features: {list(X.columns)}')
print(f'\nClass balance: Normal={( y==0).sum()} | Anomaly={(y==1).sum()}')
X.describe()

## 4. Feature Distributions by Class

In [ ]:
X_plot = X.copy()
X_plot['label'] = y.values

key_features = [
    'head_hip_dist_y', 'com_y', 'shoulder_tilt',
    'vel_com_y', 'vel_nose_y', 'hip_knee_angle_left'
]
key_features = [f for f in key_features if f in X_plot.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Feature Distributions: Normal vs. Anomaly', color='white', fontsize=13)
fig.patch.set_facecolor('#0d0d1a')

for ax, feat in zip(axes.flat, key_features):
    for lbl, label_name, color in [(0, 'Normal', PALETTE['normal']), (1, 'Anomaly', PALETTE['anomaly'])]:
        data = X_plot[X_plot['label'] == lbl][feat].dropna()
        ax.hist(data, bins=40, alpha=0.6, color=color, label=label_name, density=True)
    ax.set_title(feat, color='white', fontsize=9)
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaa')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../reports/feature_distributions.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr = X.corr()

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#1a1a2e')

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='coolwarm', center=0,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    linewidths=0.3, ax=ax,
    cbar_kws={'shrink': 0.8},
)
ax.set_title('Feature Correlation Matrix', color='white', pad=12)
ax.tick_params(colors='white', labelsize=8)

plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

## 6. Visibility Quality Analysis

In [ ]:
if 'visibility' in df.columns:
    vis_by_lm = df.groupby('landmark_id')['visibility'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(14, 4))
    fig.patch.set_facecolor('#0d0d1a')
    ax.set_facecolor('#1a1a2e')

    bars = ax.bar(
        vis_by_lm['landmark_id'],
        vis_by_lm['visibility'],
        color=[
            '#4CAF50' if v >= 0.7 else ('#FF9800' if v >= 0.4 else '#F44336')
            for v in vis_by_lm['visibility']
        ],
        edgecolor='none',
    )
    ax.axhline(0.5, color='#FF9800', linestyle='--', lw=1, label='Threshold 0.5')
    ax.set_xlabel('Landmark ID', color='#aaa')
    ax.set_ylabel('Mean Visibility', color='#aaa')
    ax.set_title('Mean Visibility Score per Landmark', color='white')
    ax.tick_params(colors='#aaa')
    ax.legend()

    low_vis = vis_by_lm[vis_by_lm['visibility'] < 0.5]
    if not low_vis.empty:
        print(f'Low visibility landmarks (< 0.5): {low_vis["landmark_id"].tolist()}')

    plt.tight_layout()
    plt.savefig('../reports/landmark_visibility.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
    plt.show()
else:
    print('No visibility column in data — skipping.')